# 🎯 LeadGen AI — Google Colab 12GB RAM Cloud Backend Runner
Ye notebook aapke LeadGen scraper ko Google Colab ki **12 GB RAM & High-Speed Cloud Network** par chalata hai.

### ✨ Latest Capabilities Included:
- **✨ LLM Query Optimization**: Raw queries (jaise `Luxury real estate developers Vasant Vihar`) ko automatically Google Maps ke liye optimize karta hai (`... in Vasant Vihar, New Delhi`).
- **🎯 Strict Locality Search**: Specific area ke bahar zoom-out ya area expansion bhatakna band; searching is locked strictly to the requested area.
- **🏁 End-of-Results Detection**: Us locality ki aakhri lead tak scroll karta hai jab tak Google Maps *"You've reached the end of the list"* na dikha de.
- **⚡ Cloudflare Tunnel**: Frontend (Vercel ya Localhost) se direct instant 1-click connect.

### 🚀 Step 1: Dependencies & Cloudflare Tunnel Install karein (1-2 Min)
Is cell ko run karke Chromium browser, Playwright-Stealth aur required packages install karein.

In [ ]:
# 1. Install Python packages (including playwright-stealth for anti-detect browsing)
!pip install -q fastapi uvicorn playwright playwright-stealth pandas openpyxl requests python-dotenv pydantic
!playwright install chromium
!playwright install-deps chromium

# 2. Download and install Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("\n✅ All Dependencies, Playwright-Stealth & Cloudflared installed successfully!")

### 📥 Step 2: Latest Code Pull karein (Smart Git Sync)
Agar pehle se clone hai to latest code pull karega, nahi to fresh clone karega.

In [ ]:
import os
if os.path.exists("/content/leadgen"):
    print("Found existing leadgen directory. Pulling latest updates from GitHub...")
    %cd /content/leadgen
    !git pull origin main
else:
    print("Cloning fresh repository from GitHub...")
    %cd /content
    !git clone https://github.com/nikhilcodeworks/leadgen.git

%cd /content/leadgen/backend
print("\n✅ Backend ready in:", !pwd)

### ⚡ Step 3: Backend Server & Cloudflare Tunnel Start karein
Ye cell chalane par aapko **trycloudflare.com** link milega. Use copy karke apne Web UI me paste karein.

In [ ]:
import os, re, subprocess, time

# Purane running processes band karein (port 8000 free karein)
!fuser -k 8000/tcp 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(1)

# Optional: Set HF Token (ya frontend UI se bhi pass kar sakte hain)
os.environ["HF_TOKEN"] = ""
os.environ["HUGGINGFACE_API_KEY"] = os.environ.get("HF_TOKEN", "")

# 1. Start FastAPI backend on port 8000
server_log = open("server.log", "w")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
server_proc = subprocess.Popen(["python", "-u", "api_server.py"], stdout=server_log, stderr=server_log, env=env)

# 2. Start Cloudflare Tunnel
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=tunnel_log,
    stderr=tunnel_log
)

print("⏳ Starting Cloudflare Tunnel... Please wait 5-10 seconds...")
tunnel_url = None
for _ in range(40):
    time.sleep(1)
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 AAPKA 12GB RAM COLAB BACKEND LIVE HAI:")
    print(f"👉 {tunnel_url}")
    print("="*70)
    print("\n📋 Next Steps:")
    print("1. Upar diya gaya trycloudflare.com URL copy karein.")
    print("2. Apne LeadGen Frontend Web Dashboard me 'Cloudflare / Colab URL' field me paste karein.")
    print("3. '⚡ Test Both' par click karein -> Connected! dikhega.")
    print("4. Ab scraping start karein — sari heavy multi-strategy scrolling aur AI evaluation Colab par chalegi!")
else:
    print("❌ Tunnel URL nahi mil saki. Logs check karein:")
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r", encoding="utf-8", errors="ignore") as f:
            print(f.read()[-500:])
    if os.path.exists("server.log"):
        with open("server.log", "r", encoding="utf-8", errors="ignore") as f:
            print(f.read()[-500:])


### 🖥️ Step 4: Live Server & Scraper Logs Monitor karein (Real-Time)
Jab aap Frontend Web UI se koi Search ya Scraping start karenge, yahan terminal par candidate discovery, LLM query optimization aur End of Results status live print hoga!

In [ ]:
import os, time

print("📡 Live Scraper Stream Active. Waiting for logs...\n" + "-"*65)
log_file = "server.log"
while not os.path.exists(log_file):
    time.sleep(1)

with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
    for line in f.readlines()[-25:]:
        print(line.rstrip())
    while True:
        line = f.readline()
        if line:
            print(line.rstrip(), flush=True)
        else:
            time.sleep(0.5)


### 🧪 Step 5 (Optional): Test Scraper Directly Inside Colab
Agar aap bina frontend ke directly Colab terminal se scraper test karna chahte hain to is cell ko run karein.

In [ ]:
# Test LLM Query Optimization & Strict Area Scraper directly:
!python -u scraper.py "Luxury real estate developers Vasant Vihar" --limit 10 --headless


### 📥 Step 6 (Optional): Download All Scraped Excel Files (.xlsx)
Agar aap saari generated Excel files apne computer par download karna chahte hain to is cell ko run karein.

In [ ]:
import glob, os
from google.colab import files

xlsx_files = glob.glob("leadsdata/*.xlsx")
if xlsx_files:
    print(f"Found {len(xlsx_files)} Excel file(s):")
    for x in xlsx_files:
        print(f" - {os.path.basename(x)}")
    !zip -q -r /content/leads_export.zip leadsdata/*.xlsx
    files.download("/content/leads_export.zip")
    print("\n✅ leads_export.zip download dialog opened!")
else:
    print("No Excel files found in leadsdata/ yet. Run a scrape first!")
